# B.2 Modellkontroll och tolkning

Denna notebook tar M2 (vald i B1) och genomför diagnostik och tolkning. M2 omskolas på hela träningsdatan (2021–2024) för att maximera precisionen i koefficientskattningarna.

**Struktur:**
1. Omskola M2 på hela träningsdatan
2. Överdispersionskontroll
3. Motivering: inga residualplottar
4. Rate ratios med konfidensintervall
5. Affärsmässig tolkning för prissättning och segmentering

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

data_dir = Path("../../../data")
df = pd.read_csv(data_dir / "Entreprenadförsäkring training.csv")

assert (df["Duration"] > 0).all(), "Duration innehåller nollor — hantera innan log"
df["Ar"] = df["Ar"].astype(int)
df["log_duration"] = np.log(df["Duration"])
df["log_omsattning"] = np.log(df["Omsattning"])

print(f"Hela träningsdatan: {df.shape[0]:,} rader (2021–2024)")

Hela träningsdatan: 1,033,386 rader (2021–2024)


## Omskola M2 på hela träningsdatan

I B1 tränades M2 på 2021–2023 med 2024 som valideringsdata för modellval. Nu när M2 är bekräftad omskolas den på alla fyra år (2021–2024) för att ge bästa möjliga koefficientskattningar inför tolkning och rapportering.

In [2]:
m2_full = smf.glm(
    "AntalSkador ~ C(Verksamhet, Treatment(reference='Byggföretag')) "
    "+ C(GeografisktOmrade, Treatment(reference='Landsbyggd')) "
    "+ log_omsattning",
    data=df,
    family=sm.families.Poisson(),
    offset=df["log_duration"],
).fit()

print(m2_full.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            AntalSkador   No. Observations:              1033386
Model:                            GLM   Df Residuals:                  1033375
Model Family:                 Poisson   Df Model:                           10
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -94353.
Date:                Thu, 23 Apr 2026   Deviance:                   1.4963e+05
Time:                        12:54:35   Pearson chi2:                 1.02e+06
No. Iterations:                     7   Pseudo R-squ. (CS):           0.005575
Covariance Type:            nonrobust                                         
                                                                                    coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------

## 1. Överdispersionskontroll

Poisson-modellen antar att variansen är lika med väntevärdet. Om Pearson-χ² / frihetsgrader är klart större än 1 tyder det på överdispersion — datan har mer variation än vad Poisson förutsäger. Det är vanligt i försäkringsdata med hög andel nollor.

In [3]:
pearson_chi2 = m2_full.pearson_chi2
df_resid = m2_full.df_resid
dispersion = pearson_chi2 / df_resid

print(f"Pearson χ²:         {pearson_chi2:,.2f}")
print(f"Frihetsgrader:      {df_resid:,}")
print(f"Dispersionskvot:    {dispersion:.4f}")

if dispersion > 1.5:
    print("\n→ Dispersionskvoten är klart större än 1 — överdispersion föreligger.")
    print("  Standardfel kan vara underskattade. Negativ binomial är ett alternativ.")
elif dispersion > 1.1:
    print("\n→ Måttlig överdispersion. Poisson-modellen är rimlig men standardfelen")
    print("  kan vara något underskattade.")
else:
    print("\n→ Dispersionskvoten ligger nära 1 — ingen allvarlig överdispersion.")

Pearson χ²:         1,018,750.47
Frihetsgrader:      1,033,375
Dispersionskvot:    0.9858

→ Dispersionskvoten ligger nära 1 — ingen allvarlig överdispersion.


### Tolkning

Med 98 % nollor i responsvariabeln är viss överdispersion förväntad. Om dispersionskvoten ligger klart över 1 innebär det att Poisson-modellens standardfel kan vara underskattade, och att konfidensintervallen bör tolkas med viss försiktighet.

**Negativ binomial** är ett alternativ som explicit hanterar överdispersion genom en extra variansparameter. Det kan nämnas som robusthetskontroll i rapporten, men Poisson-GLM behålls som huvudmodell i enlighet med projektplanen. Poisson är standard i aktuariell prissättning och ger tolkbara rate ratios.

## 2. Residualplottar

Standardresidualplottar (deviance-residualer mot anpassade värden, Q-Q-plottar) produceras **inte** i denna analys. Anledningen är att responsvariabeln `AntalSkador` är noll i cirka 98 % av observationerna. Det innebär att residualerna grupperar sig i två tydliga kluster (nollor och icke-nollor), vilket gör standardplottarna visuellt oinformativa och svåra att tolka.

Överdispersionskontrollen ovan (Pearson χ² / frihetsgrader) ger en mer relevant diagnostik för denna typ av gles count-data.

## 3. Rate ratios med konfidensintervall

Rate ratio = exp(β) anger den multiplikativa effekten på förväntad skadefrekvens jämfört med referenskategorin. Konfidensintervallen är 95 %-iga och baseras på Wald-approximationen.

In [4]:
params = m2_full.params
conf = m2_full.conf_int()

rr_table = pd.DataFrame({
    "Koefficient (β)": params.values,
    "Rate ratio": np.exp(params.values),
    "KI nedre (2.5%)": np.exp(conf.iloc[:, 0].values),
    "KI övre (97.5%)": np.exp(conf.iloc[:, 1].values),
}, index=params.index)

# Rensa variabelnamn för läsbarhet
clean_names = (
    rr_table.index
    .str.replace(r"C\(Verksamhet, Treatment\(reference='Byggföretag'\)\)\[T\.", "", regex=True)
    .str.replace(r"C\(GeografisktOmrade, Treatment\(reference='Landsbyggd'\)\)\[T\.", "", regex=True)
    .str.rstrip("]")
)
rr_table.index = clean_names
rr_table.index.name = "Variabel"

display(rr_table.round(4))

,Koefficient (β),Rate ratio,KI nedre (2.5%),KI övre (97.5%)
Variabel,,,,
Intercept,-11.1146,0.0000,0.0000,0.0000
Elektriker,-0.3602,0.6975,0.6593,0.7379
Grävning & Schaktning,-0.1568,0.8549,0.8078,0.9047
Målare,-0.4517,0.6366,0.6005,0.6748
Takarbeten,0.1266,1.1350,1.0671,1.2072
VVS,0.3588,1.4316,1.3724,1.4935
Övriga specialistföretag,-0.0304,0.9701,0.9319,1.0098
Mellanstorstad,0.1851,1.2033,1.1395,1.2708
Småstad,-0.2789,0.7566,0.7110,0.8051


### Verksamhet (relativt Byggföretag)

Varje verksamhetstyp jämförs med referenskategorin **Byggföretag**. Ett rate ratio på exempelvis 1.30 innebär att den verksamheten har 30 % högre förväntad skadefrekvens än Byggföretag, allt annat lika.

Förväntad rangordning baserat på den deskriptiva analysen (A3):
- **VVS** och **Takarbeten** förväntas ha tydligt högre skadefrekvens
- **Målare** och **Elektriker** förväntas ha lägre skadefrekvens
- **Grävning & Schaktning** och **Övriga specialistföretag** förväntas ligga nära Byggföretag

### Geografiskt område (relativt Landsbyggd)

Varje geografiskt område jämförs med referenskategorin **Landsbyggd**. Frågan är om den geografiska effekten kvarstår efter kontroll för verksamhet och omsättning.

Förväntad rangordning (A3):
- **Storstad** förväntas ha klart högre skadefrekvens
- **Mellanstorstad** förväntas ligga däremellan
- **Småstad** förväntas ligga nära eller under Landsbyggd

### log(Omsättning) — effekt per fördubbling

Koefficienten β för `log_omsattning` anger förändringen i log-skadefrekvens per enhets ökning av log(Omsättning). För att göra detta tolkbart beräknas rate ratio per **fördubbling** av omsättningen:

- En fördubbling innebär att log(Omsättning) ökar med log(2) ≈ 0.693
- Rate ratio per fördubbling = exp(β × log(2))

In [5]:
beta_oms = m2_full.params["log_omsattning"]
ci_oms = m2_full.conf_int().loc["log_omsattning"]

rr_doubling = np.exp(beta_oms * np.log(2))
rr_doubling_lower = np.exp(ci_oms.iloc[0] * np.log(2))
rr_doubling_upper = np.exp(ci_oms.iloc[1] * np.log(2))

print(f"Koefficient för log(Omsättning):        {beta_oms:.4f}")
print(f"Rate ratio per fördubbling:              {rr_doubling:.4f}")
print(f"95 % KI:                                 [{rr_doubling_lower:.4f}, {rr_doubling_upper:.4f}]")
print(f"\nTolkning: En fördubbling av omsättningen hänger samman med "
      f"{(rr_doubling - 1) * 100:.1f} % högre skadefrekvens.")

Koefficient för log(Omsättning):        0.4416
Rate ratio per fördubbling:              1.3581
95 % KI:                                 [1.3451, 1.3713]

Tolkning: En fördubbling av omsättningen hänger samman med 35.8 % högre skadefrekvens.


## 4. Affärsmässig tolkning

Nedan översätts modellens rate ratios till slutsatser som är relevanta för prissättning och segmentering.

### Verksamhetstyp som ratingfaktor

Verksamhetstyp är en av de starkaste förklarande variablerna. Skillnaden mellan den verksamhet med högst och lägst rate ratio ger spridningen i risknivå som kan motivera differentierad prissättning:

- **Högre risk:** Verksamheter med rate ratio klart över 1 relativt Byggföretag (förväntat: VVS, Takarbeten) bör prissättas med ett påslag.
- **Lägre risk:** Verksamheter med rate ratio under 1 (förväntat: Målare, Elektriker) kan motivera lägre premie.
- Spridningen mellan högsta och lägsta verksamhet ger ett mått på hur stor differentieringen bör vara.

### Geografi som ratingfaktor

Den geografiska effekten visar om lokalisering påverkar skadefrekvensen efter kontroll för verksamhet och storlek. Om skillnaden mellan Storstad och Landsbyggd är tillräckligt stor (exempelvis > 20 %) kan det motivera geografisk differentiering i prissättningen.

### Omsättning som ratingfaktor

Omsättning fångar företagsstorlek utöver vad försäkringstiden (`Duration`) redan mäter. Ett positivt samband innebär att större företag har proportionellt fler skador — premien bör alltså skala mer än linjärt med omsättning.

Log-transformen innebär att den marginella effekten avtar: skillnaden i skadefrekvens mellan ett företag med 5 och 10 MSEK omsättning är proportionellt lika stor som mellan 50 och 100 MSEK (samma fördubbling, samma rate ratio). Det är affärsmässigt rimligt — de första miljonerna i projektvolym tillför mer marginalrisk än ytterligare miljoner i redan stora verksamheter.

## 5. Sammanfattning

- **M2 bekräftad** som primär Poisson-GLM i B1 (lägst AIC och valideringsdeviance).
- **Överdispersion** kontrollerad — Poisson behålls som huvudmodell, negativ binomial nämns som alternativ.
- **Rate ratios** visar tydliga skillnader mellan verksamhetstyper och geografiska områden, samt ett positivt storlekssamband via omsättning.
- **Alla tre variabelgrupper** (verksamhet, geografi, omsättning) är relevanta som ratingfaktorer i en prissättningsmodell.

**Nästa steg:**
- Fas C: XGBoost som challenger-modell med samma variabler och valideringslogik
- Fas D: Modelljämförelse på testdata 2025
- Fas E: Osäkerhetsanalys på rad- och portföljnivå